# 🏡 Predicting House Prices with Multiple Linear Regression & Regularization
**Oasis Infobyte Internship — Data Analytics Track (Level 2 • Task 1)**  
**Author:** Narendra  
**Repository:** `OIBSIP_DataAnalytics_4`

---

## 📌 1. Project Overview & Econometric Formulation
Real estate valuation depends on a complex interplay of structural attributes and geographic premiums. This project formulates, trains, and validates a **Multiple Linear Regression** model alongside regularized variants (**Ridge $L_2$** and **Lasso $L_1$**) to accurately forecast property transaction prices.


In [ ]:
# Environment Setup & Dynamic Imports
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add paths for robust library and algorithm access
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('..'))

try:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LinearRegression, Ridge, Lasso
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
except Exception:
    from ml_core import (
        LinearRegression, Ridge, Lasso, train_test_split,
        mean_squared_error, mean_absolute_error, r2_score
    )

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

print("Dependencies successfully initialized.")


## 🔍 2. Exploratory Data Analysis & Target Variable Inspection
Loading housing transactions, checking completeness, and analyzing the distribution of property prices.


In [ ]:
# Robust path resolution
data_file = 'housing_data.csv'
possible_paths = [
    os.path.join('data', data_file),
    os.path.join('..', 'data', data_file),
    data_file
]
data_path = next((p for p in possible_paths if os.path.exists(p)), 'data/housing_data.csv')
print(f"Loading data from: {data_path}")

df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]} properties across {df.shape[1]} columns.\n")
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
# Descriptive stats of features
df.describe().T


## ⚙️ 3. Data Preprocessing & Categorical Encoding
Imputing minimal missing values with median and transforming `Neighborhood` using One-Hot Encoding (`drop_first=True` to prevent multicollinearity).


In [ ]:
df['LotArea'] = df['LotArea'].fillna(df['LotArea'].median())
df['GarageCars'] = df['GarageCars'].fillna(df['GarageCars'].median())

df_encoded = pd.get_dummies(df.drop(columns=['Property_ID']), columns=['Neighborhood'], drop_first=True, dtype=float)
df_encoded.head()


## 📊 4. Correlation Heatmap Analysis
Examining linear dependencies between structural/neighborhood variables and the target `Price`.


In [ ]:
plt.figure(figsize=(12, 8))
corr = df_encoded.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix with Housing Sale Price', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


## 🎯 5. Model Training & Regularization Comparison
Partitioning dataset into an 80% training set and 20% test holdout set, then training:
1. **Ordinary Least Squares (OLS)**
2. **Ridge Regression ($L_2$)**
3. **Lasso Regression ($L_1$)**


In [ ]:
X = df_encoded.drop(columns=['Price'])
y = df_encoded['Price']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit models
lr = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)

# Predictions
y_pred_lr = lr.predict(X_test)
y_pred_rd = ridge.predict(X_test)
y_pred_ls = lasso.predict(X_test)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

res = []
for name, preds in [('Linear Regression (OLS)', y_pred_lr), ('Ridge (L2)', y_pred_rd), ('Lasso (L1)', y_pred_ls)]:
    mae, rmse, r2 = evaluate(y_test, preds)
    res.append({'Model': name, 'MAE ($)': f"{mae:,.2f}", 'RMSE ($)': f"{rmse:,.2f}", 'R² Score': f"{r2:.4f}"})

pd.DataFrame(res)


## 📈 6. Model Evaluation & Residual Diagnostics
Visualizing predictive accuracy and testing classical linear regression assumptions.


In [ ]:
# Parity Plot
plt.figure(figsize=(9, 6))
sns.scatterplot(x=y_test, y=y_pred_lr, color='#2980b9', alpha=0.7, edgecolor='k', s=50)
min_val = min(y_test.min(), y_pred_lr.min())
max_val = max(y_test.max(), y_pred_lr.max())
plt.plot([min_val, max_val], [min_val, max_val], color='#e74c3c', linestyle='--', linewidth=2.5, label='Parity Line (y = x)')
plt.title(f'Actual vs. Predicted House Prices (R² = {r2_score(y_test, y_pred_lr):.4f})', fontsize=13, fontweight='bold')
plt.xlabel('Actual Sale Price ($)', fontsize=11)
plt.ylabel('Predicted Sale Price ($)', fontsize=11)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Residual Diagnostics
residuals = y_test - y_pred_lr
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=y_pred_lr, y=residuals, color='#8e44ad', alpha=0.7, s=50, ax=axes[0])
axes[0].axhline(0, color='#e74c3c', linestyle='--', linewidth=2)
axes[0].set_title('Residuals vs. Fitted Values (Homoscedasticity)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Sale Price ($)', fontsize=11)
axes[0].set_ylabel('Residual Error ($)', fontsize=11)

sns.histplot(residuals, kde=True, color='#16a085', ax=axes[1])
axes[1].set_title('Residual Error Distribution (Normality)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual Error ($)', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
coef_series = pd.Series(lr.coef_, index=feature_names).sort_values()
plt.figure(figsize=(11, 6))
colors = ['#e74c3c' if c < 0 else '#27ae60' for c in coef_series.values]
sns.barplot(x=coef_series.values, y=coef_series.index, palette=colors, hue=coef_series.index, legend=False)
plt.title('Marginal Feature Impact on Property Valuation ($)', fontsize=13, fontweight='bold', pad=10)
plt.xlabel('Coefficient ($ Impact per Unit Increase)', fontsize=11)
plt.ylabel('Feature', fontsize=11)

for p in plt.gca().patches:
    val = p.get_width()
    ha = 'left' if val >= 0 else 'right'
    offset = 5 if val >= 0 else -5
    plt.gca().annotate(f"${val:,.0f}", (val, p.get_y() + p.get_height()/2),
                       ha=ha, va='center', xytext=(offset, 0), textcoords='offset points', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()
